# Model ENSO full 12-2022-holdconstant

December 2022, January 2023, February 2025

Caroline Juang, c.juang@columbia.edu

**Inputs:**
* SST gradient (1982-present) - build it following the README
* trained models `Model_ENSOclim_Akaike` (SST gradient -> climate)
* trained models `Model_Akaike` (climate -> burned area)

**Variables**
* `sstpred` generally refers to the detrended SST gradient, which is the detrended-SST gradient scenario put into the climate->burned area model.
* `climpred` generally refers to the observed SST gradient, put into the climate->burned area model.

**Modified model**
* Replace parts of the trained model with constants, for these experiments:
* 1: warming variables only
    * forest: hold all non-Tmax and non-VPD variables constant.
    * non-forest: hold all non-Tmax, non-VPD, and non-current year prec constant.
* 2: prior wetting vars only
    * forest and non-forest: hold all but non- prior year prec constant.
* 3: y0 wetting variables only
    * forest and non-forest: hold all but current year prec constant.

**Outputs:**
* Predicted climate (1983-present) (but only for the relevant variables to the climate->burned area model)
* Predicted burned area (1984-present) from the trended and detrended scenarios into the climate->burned area model.
* Predicted burned area, detrended. `wumiDT`.

Data source:
* NOAA sea surface temperature, https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html

save and load models

Machine Learning Mastery: https://machinelearningmastery.com/save-load-machine-learning-models-python-scikit-learn/

In [1]:
# import
from customconfig import *
from customscripts import *

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import joblib

MANUALLY DEFINED, no forest models:
['ecoprov1', 'ecoprov3', 'ecoprov7', 'ecoprov18', 'ecoprov21']
MANUALLY DEFINED, no nonforest models:
[]
Ensemble is 500 members


In [2]:
# customize seasons for climate variables

time_length = int(finalyear-firstyear+1) # get length of timeseries
# translate years into dates
firsttime = str(firstyear)+'-01-01'
finaltime = str(finalyear)+'-12-31'

# Read the climate index from the .txt file
with open('0_climindname.txt', 'r') as f:
    climindname = f.read().strip()
print(climindname +' will be used for the SST gradient')
climindnameDT = 'DeTrend_'+climindname
climfilename = 'DeTrendClimObs_'+climindname # sst-predicted BA

# importing data string
#data_string = 'data\\'
data_string = 'data//'
#model_string = 'model\\'+climindname+'//'
model_string = 'model//'+climindname+'//'
#predict_string = 'predicted\\'+climindname+'//'
predict_string = 'predicted//'+climindname+'//'

# folder for saving figures (from customconfig)

patch125-155_nino3-34 will be used for the SST gradient


In [3]:
# manual hold variables constant (2/10/2025)

# REMOVE ALL BUT y0 TMAX AND VPD VARS (for forest)
clim_warmingfor = [
                 'solar y0', 'wind y0', 'tmin y0', 'wetdays y0',
                 'rh y0', 'rh y-1', 'prec y0', 'prec y-1',
                 'wind y-1', 'tmax y-1','tmin y-1',
                 'wetdays y-1','solar y-1','vpd y-1']
# REMOVE ALL but y0 TMAX, VPD, PREC (for non-forest)
clim_warmingnon = [
                 'solar y0', 'wind y0', 'tmin y0', 'wetdays y0',
                 'rh y0', 'rh y-1', 'prec y-1',
                 'wind y-1', 'tmax y-1','tmin y-1',
                 'wetdays y-1','solar y-1','vpd y-1']

# REMOVE ALL BUT y-1 PREC
clim_priorwetting = [
                 'solar y0', 'wind y0', 'tmax y0', 'tmin y0', 'wetdays y0',
                 'rh y0', 'rh y-1', 'prec y0', 'vpd y0',
                 'wind y-1', 'tmax y-1','tmin y-1',
                 'wetdays y-1','solar y-1','vpd y-1']
# REMOVE ALL BUT y-0 PREC
clim_currwetting = [
                 'solar y0', 'wind y0', 'tmax y0', 'tmin y0', 
                 'wetdays y0', 'rh y0', 'vpd y0',
                 'rh y-1', 'prec y-1',
                 'wind y-1', 'tmax y-1','tmin y-1',
                 'wetdays y-1','solar y-1','vpd y-1']

# compile these experiments into a bigger df, with their names
experimentnames = ['warmingvarsonly', 'prioryrwettingonly', 'y0wettingonly']
experimentfor_list = [clim_warmingfor, clim_priorwetting, clim_currwetting]
experimentnon_list = [clim_warmingnon, clim_priorwetting, clim_currwetting]

In [4]:
# get the SST inputs needed
def sortInputStrings(inputlist, istart, iend):
    """
    Requirements: 
    inputlist = list of the Model_ENSOclim_Akaike model results (locally defined)
    istart = the ecoregion name
    iend = the next ecoregion name in the list
    """
    # narrow inputs list to ecoregion
    modellistsort = inputlist[istart+1:iend+1]
    # separate concurrent and previous-year variables
    modely0names = [s for s in modellistsort if "y0" in s]
    modely1names = [s for s in modellistsort if "y-1" in s]
    return [modellistsort, modely0names, modely1names]

## Import burned area data
from `Data_CreateModelData`

In [5]:
# import observed burned area
filename = data_string + 'WUMI-ecoprovinces'
wumi = pd.read_csv(filename+'_all.txt').set_index('Unnamed: 0')
wumifor = pd.read_csv(filename+'_for.txt').set_index('Unnamed: 0')
wuminon = pd.read_csv(filename+'_non.txt').set_index('Unnamed: 0')
print('imported '+filename+' all, for, non')

imported data//WUMI-ecoprovinces all, for, non


## Import observed SST gradient data
from `Data_CreateENSOIndex` and `Data_CreateModelData`

In [6]:
# import observed SST gradient
filename = data_string + 'sstgrad_seasons_' + climindname+'_82_y.txt'
climind82_seasons = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import de-trended SST gradient
filename = data_string + 'sstgrad_seasons_' + climindnameDT+'_82_y.txt'
climind82_seasonsDT = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import observed SST gradient (with extra year, for comparison)
filename = data_string + 'sstgrad_seasons_' + climindname+'_81_y.txt'
climind81_seasons = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# IMPORT gSST AVERAGE (average of concurrent-season and prior-season)
# import observed SST gradient
filename = data_string + 'sstgrad_seasons_' + climindname+'_82_y_avgcurr-prior.txt'
climind82_seasonsavg = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# import de-trended SST gradient
filename = data_string + 'sstgrad_seasons_' + climindnameDT+'_82_y_avgcurr-prior.txt'
climind82_seasonsDTavg = pd.read_csv(filename).set_index('Unnamed: 0')
print('imported '+filename)

# put the dataframes together
climind82_seasons = pd.concat([climind82_seasons, climind82_seasonsavg], axis=1)
climind82_seasonsDT = pd.concat([climind82_seasonsDT, climind82_seasonsDTavg], axis=1)

imported data//sstgrad_seasons_patch125-155_nino3-34_82_y.txt
imported data//sstgrad_seasons_DeTrend_patch125-155_nino3-34_82_y.txt
imported data//sstgrad_seasons_patch125-155_nino3-34_81_y.txt
imported data//sstgrad_seasons_patch125-155_nino3-34_82_y_avgcurr-prior.txt
imported data//sstgrad_seasons_DeTrend_patch125-155_nino3-34_82_y_avgcurr-prior.txt


## Import observed climate data
from `Data_CreateModelData`

In [7]:
# import OBSERVED climate data to feed into models
# climate is predicted using SST, but this is the climate observed data to check

dfframesall = {}
dfframesfor = {}
dfframesnon = {}

climind = pd.read_csv(data_string + 'gradient_'+climindname+'.txt', header=None, sep=",", skiprows=[0]).set_index(0)
for iecoreg in np.arange(len(province_num)):
    filename = data_string + 'climate_ecoprovinces_'
    print(dfnames[iecoreg])
    dfframesall[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_all.txt').set_index('Unnamed: 0')
    dfframesfor[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_for.txt').set_index('Unnamed: 0')
    dfframesnon[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_non.txt').set_index('Unnamed: 0')

allwestUS
ecoprov1
ecoprov2
ecoprov3
ecoprov4
ecoprov5
ecoprov6
ecoprov7
ecoprov8
ecoprov9
ecoprov10
ecoprov12
ecoprov13
ecoprov14
ecoprov15
ecoprov16
ecoprov17
ecoprov18
ecoprov19
ecoprov20
ecoprov21


## Import model outputs

**Import the predicted climate that removes the gSST influence on climate ("sstdiff")**

from `Model_ENSOclim_Akaike`, `Model_Akaike`, and `Model_ENSOfull12-2022` (where I got the SST model, which doesn't change for these scenarios)

In [8]:
# import the climate under detrended gSST
# (already calculated as the 
# observed climate minus delta gSST contribution)

dfsstpredclimfor_sstdiff = {}
dfsstpredclimnon_sstdiff = {}
for thisname in dfnames:
    dictoutputnames = 'sstpred_for'
    filename = predict_string + 'Climate_'+thisname+'_'+dictoutputnames+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'.txt'
    dfsstpredclimfor_sstdiff[thisname] = pd.read_csv(filename).set_index('Unnamed: 0')
    dictoutputnames = 'sstpred_non'
    filename = predict_string + 'Climate_'+thisname+'_'+dictoutputnames+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'.txt'
    dfsstpredclimnon_sstdiff[thisname] = pd.read_csv(filename).set_index('Unnamed: 0')

print('imported: '+ 'Climate_'+thisname+'_'+dictoutputnames+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'.txt')

imported: Climate_ecoprov21_sstpred_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022.txt


In [9]:
# get the model input variable names from the txt files
# sst gradient to predict climate variables

with open(model_string + "modeloutput_sst_all_all.txt", "r") as f:
    inputsall = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_sst_for.txt", "r") as f:
    inputsfor = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_sst_non.txt", "r") as f:
    inputsnon = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
iinputsall = [i for i, e in 
               enumerate(inputsall) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsfor = [i for i, e in 
               enumerate(inputsfor) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsnon = [i for i, e in 
               enumerate(inputsnon) if "+++" in e]
# add in last index
iinputsall.append(len(inputsall)+1)
iinputsfor.append(len(inputsfor)+1)
iinputsnon.append(len(inputsnon)+1)

In [10]:
# get the model input variable names from the txt files
# climate variables to predict burned area

with open(model_string + "modeloutput_burnarea_for.txt", "r") as f:
    inputsfor_clim = [line.strip() for line in f]
f.close()
with open(model_string + "modeloutput_burnarea_non.txt", "r") as f:
    inputsnon_clim = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
# get ecoregions so iteration is not manual
iinputsfor_clim = [i for i, e in 
               enumerate(inputsfor_clim) if "+++" in e]
# get ecoregions so iteration is not manual
iinputsnon_clim = [i for i, e in 
               enumerate(inputsnon_clim) if "+++" in e]
# add in last index
iinputsfor_clim.append(len(inputsfor_clim)+1)
iinputsnon_clim.append(len(inputsnon_clim)+1)

# Setup for exporting burned area predictions

**500-member ensemble sequence of randomized years**
* `errormatrix` will be 500 x n-years time series of randomly-drawn years with replacement. This will be the same sequence used for both gSST-climate and climate-BA models.

**Predicted climate**
* SST_obs-predicted climate
* SST_neg-predicted climate
* difference between climate_sstobs and climate_sstneg, called climate_preddiff
* observed climate with climate_preddiff subtracted from it, called climate_sstdiff

**Predicted burned area**
* SST-predicted burned area (observed SST, goes through SST-clim and clim-BA models)
* climate-predicted burned area (observed climate, goes through clim-BA model)

In [11]:
# import the errormatrix (from Run_updateModel)
## use matrix of random years (n_years x n_ensemble members)
yearxaxis = np.arange(firstyear, finalyear+1)

tmp_fname = predict_string + "Model_ENSOFull_ErrorSequence.txt"
errormatrix = np.loadtxt(tmp_fname).astype(int) # load as integer

In [12]:
# print errormatrix to show the years
print(errormatrix[0:2,:]) # first three rows
print('\nshape: {:}'.format(np.shape(errormatrix)))

[[21  6  9  6 18 25 29 10 28  7 12 30 29 37 13 10 35  7 35 38 19 11 20  2
  13 11  7  3 18 26 29  6  2 31 16  4 20 38 33]
 [24 30  7 25 10 35  0  1 36 18 19  2  1 18 38 24  5 11 10 27 27 32  4 10
  17  8  2 14 19 36 32 27 31  6 15 21  9 29 19]]

shape: (500, 39)


# Predicted Burned Area

Iterate through each ecoprovince by `ecoregname` and `iecoreg`, calculate the predicted burned area based on the observed climate `dfclimpredBA` and based on the SST difference `dfsstpredBA`.

Calculate full WUMI without SST gradient trend (DT scenario), outputted as `BurnArea_wumiDT`

In [13]:
# a bunch of dictionaries for storage
# predicted climate from observed SST
dictclimpredBAall = {}
dictclimpredBAfor = {}
dictclimpredBAnon = {}

# predicted climate from negative-trended SST
dictsstpredBAall = {}
dictsstpredBAfor = {}
dictsstpredBAnon = {}

# difference in predicted climate scenarios, climate_preddiff
dfsstpredclimall_preddiff = {}
dfsstpredclimfor_preddiff = {}
dfsstpredclimnon_preddiff = {}

# iterate through different experiments
# export to a dict by experiment name

for j, thisexperiment in enumerate(experimentnames):

    # parse experiment list
    thisexplistfor = experimentfor_list[j]
    thisexplistnon = experimentnon_list[j]

    # PREDICT AREA BURNED, storage

    # SST-predicted, from climate_sstdiff
    dfsstpredBAall = [] # will be 500*21 x 39
    dfsstpredBAfor = []
    dfsstpredBAnon = []
    # climate-predicted, from observed climate
    dfclimpredBAall = [] # will be 21 x 39
    dfclimpredBAfor = []
    dfclimpredBAnon = []
    burnareapredloop = [] # column names storage
    
    # write the same print statements to this file
    modeloutputfile = 'Model_ENSOfull-outputsClim-BA_'+experimentnames[j]+'.txt'
    f = open(modeloutputfile, 'w') # print to this file
    import warnings
    warnings.filterwarnings('ignore')

    # get this ecoregion
    for thisi, thisname in enumerate(dfnames):
        ecoregname = thisname
        iecoreg = thisi
    
        ###########
    
        landname = 'for'
        landtypeinput = inputsfor
        landtypeiinput = iinputsfor
        landtypeinputclim = inputsfor_clim
        landtypeiinputclim = iinputsfor_clim
        # finalclimpred is fed into the burned area model
        filename = model_string + 'model_'+landname+'_burnedarea_'+ecoregname+'.sav'
        modelburnarea = joblib.load(filename)
        # single-out the variables needed for the two models
        ithisecoreg = landtypeinput.index('+++'+ecoregname)
        ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
        ithisecoreg_clim = landtypeinputclim.index('+++'+ecoregname)
        ithisecoregend_clim = landtypeiinputclim[landtypeiinputclim.index(ithisecoreg_clim)+1]-1
        [climnamessort, climy0names, climy1names] = sortInputStrings(landtypeinputclim,
                                                                    ithisecoreg_clim, 
                                                                    ithisecoregend_clim)
        # SET SOME VARIABLES CONSTANT
        settozero = np.zeros(len(climnamessort)) # storage
        settozerosort = np.zeros(len(climnamessort))
        # iterate through each var name to figure out vars to hold constant
        for xx, label in enumerate(climnamessort):
            tmplabeljustvar = label.split(' ')[0]
            tmplabeljustyr = label.split(' ')[1]
            tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
            # set to one if it's a variable to hold constant
            if tmpnameyrlabel in thisexplistfor:
                # hold it constant
                settozero[xx] = 1
        # set the loaded model coefficients to zero using settozero
        tmpcoef = modelburnarea.coef_
        tmpcoef[settozero.astype(bool)] = 0
        modelburnarea.coef_ = tmpcoef # REPLACE coefs in MODEL

        # SST-PREDICTED CLIMATE FOREST (go through 500-member)
        for k in np.arange(0,np.shape(errormatrix)[0]):
            # get all the climate members for k
            tmpcolmem = []
            for tmpcol in climnamessort:
                tmpcolmem.append(tmpcol + '_' + str(k)) # member k climate
            # grab the adjusted climate predictions, predict area burned
            Xclimpred = dfsstpredclimfor_sstdiff[ecoregname][tmpcolmem] # FOREST
            # rename columns to match the BA model
            Xclimpred.columns = climnamessort
            
            # predict area burned for detrended sst scenario
            burnareapred = modelburnarea.predict(Xclimpred) # predicted value
            # SAVE the predicted outputs
            dfsstpredBAfor.append(burnareapred) # append SST-predicted BA x500
            namepredloop = thisname + '_' + str(k) # ecoprovince_k
            burnareapredloop.append(namepredloop) # append column NAME!!!!
            
        # grab the OBSERVED CLIMATE, predict area burned
        Xclimpredobs = dfframesfor[ecoregname][climnamessort]
        print(prov_abbr_names[thisi] + ' - FOREST BA-model vars: ')
        f.write(prov_abbr_names[thisi] + ' - FOREST: \n')
        tmpvariableeq = dfframesfor[ecoregname][climnamessort].columns.values
        print('log10(BA) = ')
        f.write('log10(BA) = \n')
        for thisvari, thisvarname in enumerate(tmpvariableeq):
            tmpfullname, thisunits, thiscolor = suppclimplotformat(thisvarname)
            print('+ {:.3f}({:}) '.format(modelburnarea.coef_[thisvari], tmpfullname))
            f.write('+ {:.3f}({:}) '.format(modelburnarea.coef_[thisvari], tmpfullname))
        print('+ {:.3f} + \u03B5\n'.format(modelburnarea.intercept_))
        f.write('+ {:.3f} + \u03B5\n'.format(modelburnarea.intercept_))
        burnareapredobs = modelburnarea.predict(Xclimpredobs)
    
        # SAVE the predicted outputs
        dfclimpredBAfor.append(burnareapredobs) # append clim-predicted BA
    
        # all data
        wumicol = np.array(wumifor.iloc[:,iecoreg])
        fig, ax = plt.subplots(figsize=(6,3))
        yearxaxis = np.arange(firstyear, finalyear+1)
        ax.plot(yearxaxis, wumicol, c='k', label='Observed burned area')
        ax.plot(yearxaxis, 10**burnareapredobs, label='From climate, observed')
        ax.plot(yearxaxis, 10**burnareapred, label='From climate, no SST trend')
        ax.legend()
        tmpcorr = pearsonr(burnareapredobs, burnareapred)
        ax.set_title('Burned area in '+landname + ', '+ ecoregname+ ', \nObserved Climate vs. Climate under '+ climindnameDT);
        fig.tight_layout()
        #plt.savefig(figfolder + 'Model_ENSOfull12-2022_'+landname+'_'+ecoregname+'_BurnAreaPred')
        plt.close()
        
        ###########
    
        landname = 'non'

        landtypeinput = inputsnon
        landtypeiinput = iinputsnon
        landtypeinputclim = inputsnon_clim
        landtypeiinputclim = iinputsnon_clim
        # finalclimpred is fed into the burned area model
        filename = model_string + 'model_'+landname+'_burnedarea_'+ecoregname+'.sav'
        modelburnarea = joblib.load(filename)
        # single-out the variables needed for the two models
        ithisecoreg = landtypeinput.index('+++'+ecoregname)
        ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
        ithisecoreg_clim = landtypeinputclim.index('+++'+ecoregname)
        ithisecoregend_clim = landtypeiinputclim[landtypeiinputclim.index(ithisecoreg_clim)+1]-1
        [climnamessort, climy0names, climy1names] = sortInputStrings(landtypeinputclim,
                                                                    ithisecoreg_clim, 
                                                                    ithisecoregend_clim)
        # SET SOME VARIABLES CONSTANT
        settozero = np.zeros(len(climnamessort)) # storage
        settozerosort = np.zeros(len(climnamessort))
        # iterate through each var name to figure out vars to hold constant
        for xx, label in enumerate(climnamessort):
            tmplabeljustvar = label.split(' ')[0]
            tmplabeljustyr = label.split(' ')[1]
            tmpnameyrlabel = tmplabeljustvar + ' ' + tmplabeljustyr
            # set to one if it's a variable to hold constant
            if tmpnameyrlabel in thisexplistnon: # NONFOREST
                # hold it constant
                settozero[xx] = 1
        # set the loaded model coefficients to zero using settozero
        tmpcoef = modelburnarea.coef_
        tmpcoef[settozero.astype(bool)] = 0
        modelburnarea.coef_ = tmpcoef # REPLACE coefs in MODEL

        # SST-PREDICTED CLIMATE NONFOREST (go through 500-member)
        for k in np.arange(0,np.shape(errormatrix)[0]):
            # get all the climate members for k
            tmpcolmem = []
            for tmpcol in climnamessort:
                tmpcolmem.append(tmpcol + '_' + str(k)) # member k climate
            # grab the adjusted climate predictions, predict area burned
            Xclimpred = dfsstpredclimnon_sstdiff[ecoregname][tmpcolmem] # NONFOREST
            # rename columns to match the BA model
            Xclimpred.columns = climnamessort
            
            # predict area burned for detrended sst scenario
            burnareapred = modelburnarea.predict(Xclimpred) # predicted value
            # SAVE the predicted outputs
            dfsstpredBAnon.append(burnareapred) # append SST-predicted BA NONFOREST
    
        # grab the OBSERVED CLIMATE, predict area burned
        Xclimpredobs = dfframesnon[ecoregname][climnamessort] # NONFOREST
        print(prov_abbr_names[thisi] + ' - NONFOREST BA-model vars: ')
        f.write(prov_abbr_names[thisi] + ' - NONFOREST: \n')
        tmpvariableeq = dfframesnon[ecoregname][climnamessort].columns.values # NONFOREST
        print('log10(BA) = ')
        f.write('log10(BA) = \n')
        for thisvari, thisvarname in enumerate(tmpvariableeq):
            tmpfullname, thisunits, thiscolor = suppclimplotformat(thisvarname)
            print('+ {:.3f}({:}) '.format(modelburnarea.coef_[thisvari], tmpfullname))
            f.write('+ {:.3f}({:}) '.format(modelburnarea.coef_[thisvari], tmpfullname))
        print('+ {:.3f} + \u03B5\n'.format(modelburnarea.intercept_))
        f.write('+ {:.3f} + \u03B5\n'.format(modelburnarea.intercept_))
        burnareapredobs = modelburnarea.predict(Xclimpredobs)
    
        # SAVE the predicted outputs
        dfclimpredBAnon.append(burnareapredobs) # append clim-predicted BA NONFOREST
    
        # all data
        wumicol = np.array(wuminon.iloc[:,iecoreg])
        fig, ax = plt.subplots(figsize=(6,3))
        yearxaxis = np.arange(firstyear, finalyear+1)
        ax.plot(yearxaxis, wumicol, c='k', label='Observed burned area')
        ax.plot(yearxaxis, 10**burnareapredobs, label='From climate, observed')
        ax.plot(yearxaxis, 10**burnareapred, label='From climate, no SST trend')
        ax.legend()
        tmpcorr = pearsonr(burnareapredobs, burnareapred)
        ax.set_title('Burned area in '+landname + ', '+ ecoregname+ ', \nObserved Climate vs. Climate under '+ climindnameDT);
        fig.tight_layout()
        #plt.savefig(figfolder + 'Model_ENSOfull12-2022_'+landname+'_'+ecoregname+'_BurnAreaPred')
        plt.close()
        ############
    
    landname = 'all'
    
    # instead of using model, we sum forest+nonforest predictions
    # SAVE the predicted outputs
    dfsstpredBAall = np.log10((10**np.array(dfsstpredBAfor))+(10**np.array(dfsstpredBAnon)))
    dfclimpredBAall = np.log10((10**np.array(dfclimpredBAfor))+(10**np.array(dfclimpredBAnon)))
    
    # save the burn areas to the dict version for the experiment
    # detrended-SST version
    dictsstpredBAall[thisexperiment] = dfsstpredBAall
    dictsstpredBAfor[thisexperiment] = dfsstpredBAfor
    dictsstpredBAnon[thisexperiment] = dfsstpredBAnon
    # climate-predicted, from observed climate
    dictclimpredBAall[thisexperiment] = dfclimpredBAall
    dictclimpredBAfor[thisexperiment] = dfclimpredBAfor
    dictclimpredBAnon[thisexperiment] = dfclimpredBAnon
    # close file
    f.close()
    

0: All western US - FOREST BA-model vars: 
log10(BA) = 
+ 0.287(VPD y-0,JAS) 
+ 0.207(VPD y-0,AMJ) 
+ 0.112(Tmax y-0,OND) 
+ 3.392 + ε



0: All western US - NONFOREST BA-model vars: 


log10(BA) = 
+ 0.150(VPD y-0,JAS) 
+ 0.000(Precipitation y-1,AMJ) 
+ 3.695 + ε

1: American Semi-Desert - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-1,JFM) 
+ 1.103 + ε



1: American Semi-Desert - NONFOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-1,OND) 
+ 0.000(Precipitation y-1,JFM) 
+ 0.115(Tmax y-0,JAS) 
+ 2.038 + ε

2: AZ-NM Mountains - FOREST BA-model vars: 
log10(BA) = 
+ 0.262(VPD y-0,AMJ) 
+ 0.245(Tmax y-0,JAS) 
+ 2.284 + ε



2: AZ-NM Mountains - NONFOREST BA-model vars: 
log10(BA) = 
+ 0.000(Wind y-0,JFM) 
+ 0.814(Tmax y-0,JAS) 
+ 0.000(Wet_days y-0,JAS) 
+ 0.000(Precipitation y-1,AMJ) 
+ 1.431 + ε

x: Black Hills Coniferous Forest - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-0,AMJ) 
+ 0.322 + ε

x: Black Hills Coniferous Forest - NONFOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-1,JFM) 
+ -1.234 + ε



...

13: NV-UT Mountains - NONFOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-1,AMJ) 
+ 0.000(Tmax y-0,AMJ) 
+ 0.000(Wind y-0,JFM) 
+ 1.691 + ε



14: Northern Rocky Mountain Forest - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(VPD y-0,AMJ) 
+ 1.825 + ε

14: Northern Rocky Mountain Forest - NONFOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Precipitation y-1,JAS) 
+ 0.000(VPD y-0,JFM) 
+ 1.430 + ε

x: Pacific Lowland Mixed Forest Province - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Tmax y-0,OND) 
+ -0.416 + ε



x: Pacific Lowland Mixed Forest Province - NONFOREST BA-model vars: 
log10(BA) = 
+ 0.000(Wet_days y-0,JAS) 
+ -1.049 + ε

15: Sierran Steppe - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Tmax y-0,AMJ) 
+ 0.000(Precipitation y-1,JFM) 
+ 0.000(VPD y-0,OND) 
+ 2.595 + ε



15: Sierran Steppe - NONFOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-1,JFM) 
+ 0.000(VPD y-0,AMJ) 
+ 0.000(VPD y-0,OND) 
+ 2.251 + ε

16: Southern Rocky Mountain Steppe - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,JAS) 
+ 0.000(Solar_radiation y-0,JFM) 
+ 0.000(Wind y-0,JFM) 
+ 2.093 + ε

16: Southern Rocky Mountain Steppe - NONFOREST BA-model vars: 
log10(BA) = 
+ 0.000(VPD y-0,AMJ) 
+ 0.000(Solar_radiation y-0,JAS) 
+ 1.770 + ε



17: SW Plateau and Plains - FOREST BA-model vars: 
log10(BA) = 
+ 0.000(Tmax y-0,JFM) 
+ -1.459 + ε

17: SW Plateau and Plains - NONFOREST BA-model vars: 
log10(BA) = 
+ 0.000(Precipitation y-1,JAS) 
+ 0.000(Wind y-0,JFM) 
+ 0.000(Precipitation y-1,JFM) 
+ 0.000(Solar_radiation y-0,OND) 
+ 0.000(Precipitation y-1,AMJ) 
+ 1.712 + ε



## set Ecoprovince 0 (all west US) as sum of the ecoprovinces

Iterate through each experiment to do this

In [14]:
# make pandas df at this point for every experiment
for j, thisexperiment in enumerate(experimentnames):
    dfsstpredBAall_df = pd.DataFrame(np.transpose(np.stack(dictsstpredBAall[thisexperiment])), # sst-pred 
                                 columns=[burnareapredloop], index=yearxaxis)
    dfsstpredBAfor_df = pd.DataFrame(np.transpose(np.stack(dictsstpredBAfor[thisexperiment])), 
                                     columns=[burnareapredloop], index=yearxaxis)
    dfsstpredBAnon_df = pd.DataFrame(np.transpose(np.stack(dictsstpredBAnon[thisexperiment])), 
                                     columns=[burnareapredloop], index=yearxaxis)
    dfclimpredBAall_df = pd.DataFrame(np.transpose(np.stack(dictclimpredBAall[thisexperiment])), # obsclim-pred
                                     columns=[dfnames], index=yearxaxis)
    dfclimpredBAfor_df = pd.DataFrame(np.transpose(np.stack(dictclimpredBAfor[thisexperiment])), 
                                     columns=[dfnames], index=yearxaxis)
    dfclimpredBAnon_df = pd.DataFrame(np.transpose(np.stack(dictclimpredBAnon[thisexperiment])), 
                                     columns=[dfnames], index=yearxaxis)

    print('output for experiment ['+thisexperiment+']')
    # SUM ECOPROVINCES FOR ALL WESTERN US
    # make everything unlogged
    # then sum the burned area from all ecoprovinces
    # then log it again, and set the first ecoprovince as the sums

    # sst-pred
    for k in range(500):
        # select all ecoprov columns for this k; FOREST + NONFOREST
        cols_k = [c for c in burnareapredloop if c.endswith(f"_{k}") and "ecoprov" in c]
        # log10 to linear
        tmplinearBA = 10 ** dfsstpredBAall_df[cols_k]
        # sum across ecoprovinces (axis=1 = across columns)
        tmpsummed = tmplinearBA.sum(axis=1)
        # back to log10
        dfsstpredBAall_df[f"allwestUS_{k}"] = np.log10(tmpsummed)
    
        # FOREST
        cols_k = [c for c in burnareapredloop if c.endswith(f"_{k}") and "ecoprov" in c]
        # log10 to linear
        tmplinearBA = 10 ** dfsstpredBAfor_df[cols_k]
        # sum across ecoprovinces (axis=1 = across columns)
        tmpsummed = tmplinearBA.sum(axis=1)
        # back to log10
        dfsstpredBAfor_df[f"allwestUS_{k}"] = np.log10(tmpsummed)
    
        # NON-FOREST
        cols_k = [c for c in burnareapredloop if c.endswith(f"_{k}") and "ecoprov" in c]
        # log10 to linear
        tmplinearBA = 10 ** dfsstpredBAnon_df[cols_k]
        # sum across ecoprovinces (axis=1 = across columns)
        tmpsummed = tmplinearBA.sum(axis=1)
        # back to log10
        dfsstpredBAnon_df[f"allwestUS_{k}"] = np.log10(tmpsummed)

    # clim-pred
    tmplinearBA = 10**dfclimpredBAall_df # unlog
    dfclimpredBAall_df['allwestUS'] = np.log10((tmplinearBA.iloc[:,1:]).sum(axis=1)) # re-log
    
    tmplinearBA = 10**dfclimpredBAfor_df # unlog
    dfclimpredBAfor_df['allwestUS'] = np.log10((tmplinearBA.iloc[:,1:]).sum(axis=1)) # re-log
    
    tmplinearBA = 10**dfclimpredBAnon_df # unlog
    dfclimpredBAnon_df['allwestUS'] = np.log10((tmplinearBA.iloc[:,1:]).sum(axis=1)) # re-log
     

    # ------------- Calculate WUMI without SST gradient trend
    # ------------- exported as BurnArea_wumiDT
    counter = 0 # for wumi
    wumiallDT = []
    wumiforDT = []
    wuminonDT = []
    
    for thisname in dfnames:
        # grab values
        tmpBA = wumi[str(counter)].values
        tmpBAfor = wumifor[str(counter)].values
        tmpBAnon = wuminon[str(counter)].values
        tmpBAclim = 10**dfclimpredBAall_df[thisname].values.flatten() # unlog
        tmpBAclimfor = 10**dfclimpredBAfor_df[thisname].values.flatten()
        tmpBAclimnon = 10**dfclimpredBAnon_df[thisname].values.flatten()
        # grab values
        for k in range(500):
            # select ecoprovince + k
            thisnamek = thisname +'_'+str(k)
            tmpBAsst = 10**dfsstpredBAall_df[thisnamek].values.flatten() # unlog
            tmpBAsstfor = 10**dfsstpredBAfor_df[thisnamek].values.flatten()
            tmpBAsstnon = 10**dfsstpredBAnon_df[thisnamek].values.flatten()
    
            # calculate the full BA without SST gradient trend influence
            # observed BA - (observed SST in clim-BA model BA - DT SST clim-BA model BA)
            tmpBAnosst = tmpBA - (tmpBAclim - tmpBAsst)
            tmpBAnosstfor = tmpBAfor - (tmpBAclimfor - tmpBAsstfor)
            tmpBAnosstnon = tmpBAnon - (tmpBAclimnon - tmpBAsstnon)
    
            wumiallDT.append(tmpBAnosst)
            wumiforDT.append(tmpBAnosstfor)
            wuminonDT.append(tmpBAnosstnon)
    
    # transform into pandas dataframe
    wumiallDT = pd.DataFrame(np.transpose(np.stack(wumiallDT)),
                             columns=[burnareapredloop], index=yearxaxis)
    wumiforDT = pd.DataFrame(np.transpose(np.stack(wumiforDT)),
                             columns=[burnareapredloop], index=yearxaxis)
    wuminonDT = pd.DataFrame(np.transpose(np.stack(wuminonDT)),
                             columns=[burnareapredloop], index=yearxaxis)


    # ------------ export data by experiment
    
    dictoutputs = [dfsstpredBAall_df, dfsstpredBAfor_df, dfsstpredBAnon_df,
                   dfclimpredBAall_df, dfclimpredBAfor_df, dfclimpredBAnon_df,
                   wumiallDT, wumiforDT, wuminonDT]
    dictoutputnames = ['sstpred_all', 'sstpred_for', 'sstpred_non',
                       'climpred_all','climpred_for','climpred_non',
                       'wumiDT_all', 'wumiDT_for', 'wumiDT_non']
    
    # convert to pandas df and export
    for i,thisdict in enumerate(dictoutputs):
        tmpoutput = thisdict
        if i<3: # export the SST-predicted
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        if (i>=3) & (i<6): # export the climate-predicted
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        if (i>=6): # export the detrended wumi
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        print('Exported: '+filename)

output for experiment [warmingvarsonly]


Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_all_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_for_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_non_1984-2022_warmingvarsonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt


...

Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_all_1984-2022_y0wettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_for_1984-2022_y0wettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_non_1984-2022_y0wettingonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt


# Ensemble Error

Replaces the current `wumifor`, `wuminon`, `dfclimpredBAfor`, and `dfclimpredBAnon`

September 4, 2024

Because our model predicts log10(BA), it is prone to errors at higher burned area values. In general, by predicting log10(BA), our model will tend to underestimate . When we take the mean of log10(BA) for the entire timeseries, then the mean would be an underestimate of the actual mean. (the mean of 10^1, 10^2, and 10^3 would be (1+2+3)/3, which is 2, but that would be 100. But the real average would be 370.

* BA_mod = modeled burned area
* BA_obs = observed burned area

Method is as follows:
1. Calculate the time series of errors `BA_error_log10 = log10(BA_obs) - log10(BA_mod)`
3. make an ensemble of 500 time series of `log10(BA_mod_plus_error)`.
    * for each step in the time series of `BA_mod`, the error is randomly selected and added to the step in the time series by sampling from the time series of `BA_error_log10` with replacement, not including sampling from the index year being selected for.
    * Result is one 39x500 matrix (years x ensembles) of randomly-selected years is created, and used to pick for all of the ecoprovinces and timeseries at once.
4. Un-log all 500 time series of log10(BA_mod_plus_error) to get the 500-member ensemble in original km^2 units.
5. Convert negative values to 0 km^2
6. Take the average of all 500 time series.

In [15]:
# import OBSERVED burned area wumi
filename = data_string + 'WUMI-ecoprovinces'
wumi = pd.read_csv(filename+'_all.txt').set_index('Unnamed: 0')
wumifor = pd.read_csv(filename+'_for.txt').set_index('Unnamed: 0')
wuminon = pd.read_csv(filename+'_non.txt').set_index('Unnamed: 0')

# import DETRENDED burned area wumi
filename = predict_string + 'BurnArea_wumiDT'
filenameend = '_'+str(firstyear)+'-'+str(finalyear)+'.txt'
wumiDT = pd.read_csv(filename+'_all_'+climfilename+filenameend).set_index('Unnamed: 0')
wumiforDT = pd.read_csv(filename+'_for_'+climfilename+filenameend).set_index('Unnamed: 0')
wuminonDT = pd.read_csv(filename+'_non_'+climfilename+filenameend).set_index('Unnamed: 0')

#filename = predict_string + 'BurnArea_'
filename = predict_string + 'BurnArea_'

# predicted burned area from observed climate (FROM Model_ENSOfull)
BAclimpredall = pd.read_csv(filename+'climpred_all_'+str(firstyear)+'-'+str(finalyear)+'.txt').set_index('Unnamed: 0')
BAclimpredfor = pd.read_csv(filename+'climpred_for_'+str(firstyear)+'-'+str(finalyear)+'.txt').set_index('Unnamed: 0')
BAclimprednon = pd.read_csv(filename+'climpred_non_'+str(firstyear)+'-'+str(finalyear)+'.txt').set_index('Unnamed: 0')

# import the errormatrix (from Run_updateModel)
## use matrix of random years (n_years x n_ensemble members)
yearxaxis = np.arange(firstyear, finalyear+1)
tmp_fname = predict_string + "Model_ENSOFull_ErrorSequence.txt"
errormatrix = np.loadtxt(tmp_fname).astype(int) # load as integer

In [16]:
# log all wumi values
# BAclimpred is already log
BA_obsfor = np.ma.masked_invalid(np.log10(wumifor)).filled(0)
BA_obsnon = np.ma.masked_invalid(np.log10(wuminon)).filled(0)
BA_obsfor = pd.DataFrame(BA_obsfor, index=yearxaxis, columns=dfnames)
BA_obsnon = pd.DataFrame(BA_obsnon, index=yearxaxis, columns=dfnames)

# get log difference BA_obs - BA_model
BA_errorfor_log10 = BA_obsfor - BAclimpredfor
BA_errornon_log10 = BA_obsnon - BAclimprednon

In [17]:
# import the BA predicted from the scenarios, arranged 
# into the dict format

dictBAclimpredall_DT = {} # import variables into these storages
dictBAclimpredfor_DT = {}
dictBAclimprednon_DT = {}
dictBAclimpredall = {}
dictBAclimpredfor = {}
dictBAclimprednon = {}
dictwumiallDT = {}
dictwumiforDT = {}
dictwuminonDT = {}

# PREDICTED BURNED AREA from observed SST and observed climate 
# (Model_ENSOfull12-2022-holdconstant must be updated)

for j, thisexperiment in enumerate(experimentnames):
    print('reimport experiment ['+thisexperiment+']')
    dictoutputnames = ['sstpred_all', 'sstpred_for', 'sstpred_non',
                       'climpred_all','climpred_for','climpred_non']
    filename = predict_string + 'BurnArea_sstpred_all_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    
    # predicted burned area removing SST gradient trend
    # (observed climate - climate without effect of SST trend)
    dictBAclimpredall_DT[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    filename = predict_string + 'BurnArea_sstpred_for_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    print(filename)
    dictBAclimpredfor_DT[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    filename = predict_string + 'BurnArea_sstpred_non_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictBAclimprednon_DT[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    
    # predicted burned area from observed climate
    filename = predict_string + 'BurnArea_climpred_all_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictBAclimpredall[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    filename = predict_string + 'BurnArea_climpred_for_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictBAclimpredfor[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')
    filename = predict_string + 'BurnArea_climpred_non_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment
    dictBAclimprednon[thisexperiment] = pd.read_csv(filename+'.txt').set_index('Unnamed: 0')

reimport experiment [warmingvarsonly]
predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly


reimport experiment [prioryrwettingonly]
predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_prioryrwettingonly


reimport experiment [y0wettingonly]
predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly


In [18]:
## ITERATE THROUGH EXPERIMENTS, CALCULATE BA + ERROR

for j,thisexperiment in enumerate(experimentnames):
    # EXPORT CLIMATE-PREDICTED (SST TRENDED) AND
    # SST-PREDICTED (SST-DETRENDED) MODELED BA WITH ADDED LOG(BA) ERRORS (redone 4/30/2026)

    # iterate through each ecoprovince
    # storage for final
    BA_mod_plus_errorfor = []
    BA_mod_plus_errornon = []
    BA_modDT_plus_errorfor = []
    BA_modDT_plus_errornon = []
    nameslist = [] # for storing all names formatted "ecoprov_k"

    # for one ecoregion
    # pick ecoregion name, select columns
    for thisname in dfnames:
        # grab values (clim pred)
        # start with thisexperiment
        tmpBAmodfor = dictBAclimpredfor[thisexperiment][thisname].values.flatten() # trended SST
        tmpBAmodnon = dictBAclimprednon[thisexperiment][thisname].values.flatten()
    
        for k in range(len(errormatrix)):
            # grab values (sst pred)
            # select ecoprovince + k, start with thisexperiment
            thisnamek = thisname +'_'+str(k)
            nameslist.append(thisnamek) # store
            tmpBAmodDTfor = dictBAclimpredfor_DT[thisexperiment].loc[:,thisnamek] # detrended SST
            tmpBAmodDTnon = dictBAclimprednon_DT[thisexperiment].loc[:,thisnamek]
            
            # get error for this ecoprovince and this configuration
            tmperrorfor = BA_errorfor_log10[thisname].values[errormatrix[k]]
            tmperrornon = BA_errornon_log10[thisname].values[errormatrix[k]]
            
            # BA plus error, then unlog
            BA_mod_plus_errorfor.append(10**(tmpBAmodfor + tmperrorfor))
            BA_mod_plus_errornon.append(10**(tmpBAmodnon + tmperrornon))
            BA_modDT_plus_errorfor.append(10**(tmpBAmodDTfor + tmperrorfor))
            BA_modDT_plus_errornon.append(10**(tmpBAmodDTnon + tmperrornon))

    # create dataframe
    dfBA_mod_plus_errorfor = pd.DataFrame(np.transpose(BA_mod_plus_errorfor), 
                                            columns=nameslist, index=yearxaxis)
    dfBA_mod_plus_errornon = pd.DataFrame(np.transpose(BA_mod_plus_errornon), 
                                            columns=nameslist, index=yearxaxis)
    dfBA_modDT_plus_errorfor = pd.DataFrame(np.transpose(BA_modDT_plus_errorfor), 
                                            columns=nameslist, index=yearxaxis)
    dfBA_modDT_plus_errornon = pd.DataFrame(np.transpose(BA_modDT_plus_errornon), 
                                            columns=nameslist, index=yearxaxis)
    
    # RULE: set negative values to zero
    dfBA_mod_plus_errorfor[dfBA_mod_plus_errorfor<0] = 0
    dfBA_mod_plus_errornon[dfBA_mod_plus_errornon<0] = 0
    dfBA_modDT_plus_errorfor[dfBA_modDT_plus_errorfor<0] = 0
    dfBA_modDT_plus_errornon[dfBA_modDT_plus_errornon<0] = 0
    
    # NOTE: AT THE END OF THIS, BA IS LINEAR
    
    # --------------------------------------

    # # SUM ECOPROVINCES FOR ALL WESTERN US
    # then sum the burned area from all ecoprovinces
    # then log it again, and set the first ecoprovince as the sums

    # sst-pred
    for k in range(len(errormatrix)):
        # CLIM-PRED
        # select all ecoprov columns for this k
        cols_k = [c for c in nameslist if c.endswith(f"_{k}") and "ecoprov" in c]
        
        # FOREST; while BA is linear, sum across ecoprovinces (axis=1 = across columns)
        tmplinearBA = (dfBA_mod_plus_errorfor[cols_k]).sum(axis=1)
        dfBA_mod_plus_errorfor[f"allwestUS_{k}"] = tmplinearBA
        # NONFOREST; while BA is linear, sum across ecoprovinces (axis=1 = across columns)
        tmplinearBA = (dfBA_mod_plus_errornon[cols_k]).sum(axis=1)
        dfBA_mod_plus_errornon[f"allwestUS_{k}"] = tmplinearBA
        
        
        # SST-PRED
        # FOREST; while BA is linear, sum across ecoprovinces (axis=1 = across columns)
        tmplinearBA = (dfBA_modDT_plus_errorfor[cols_k]).sum(axis=1)
        dfBA_modDT_plus_errorfor[f"allwestUS_{k}"] = tmplinearBA
        # NONFOREST; while BA is linear, sum across ecoprovinces (axis=1 = across columns)
        tmplinearBA = (dfBA_modDT_plus_errornon[cols_k]).sum(axis=1)
        dfBA_modDT_plus_errornon[f"allwestUS_{k}"] = tmplinearBA
        
    # convert all to log(BA)
    dfBA_mod_plus_errorfor = np.log10(dfBA_mod_plus_errorfor)
    dfBA_mod_plus_errornon = np.log10(dfBA_mod_plus_errornon)
    dfBA_modDT_plus_errorfor = np.log10(dfBA_modDT_plus_errorfor)
    dfBA_modDT_plus_errornon = np.log10(dfBA_modDT_plus_errornon)
    
    # sum forest and nonforest to get "all"
    dfBA_mod_plus_errorall = np.log10((10**dfBA_mod_plus_errorfor) + (10**dfBA_mod_plus_errornon))
    dfBA_modDT_plus_errorall = np.log10((10**dfBA_modDT_plus_errorfor) + (10**dfBA_modDT_plus_errornon))
    
    # NOTE: AT THE END OF THIS, BA IS LOG(BA)

    # -----------------------

    # prep for export, reassign to variable names
    dfclimpredBAall = dfBA_mod_plus_errorall
    dfclimpredBAfor = dfBA_mod_plus_errorfor
    dfclimpredBAnon = dfBA_mod_plus_errornon
    
    dfsstpredBAall = dfBA_modDT_plus_errorall
    dfsstpredBAfor = dfBA_modDT_plus_errorfor
    dfsstpredBAnon = dfBA_modDT_plus_errornon

    # -----------------------

    # Calculate full WUMI without SST gradient trend + error (DT scenario)
    # outputted as `BurnArea_wumiDT`, with new BA+error

    counter = 0 # for wumi
    wumiallDT = []
    wumiforDT = []
    wuminonDT = []
    nameslist = []

    for thisname in dfnames:
        # grab values
        tmpBA = wumi[str(counter)].values
        tmpBAfor = wumifor[str(counter)].values
        tmpBAnon = wuminon[str(counter)].values
    
        # grab values
        for k in range(500):
            # select ecoprovince + k
            thisnamek = thisname +'_'+str(k)
            nameslist.append(thisnamek)
    
            tmpBAclim = 10**dfBA_mod_plus_errorall[thisnamek].values.flatten() # unlog
            tmpBAclimfor = 10**dfBA_mod_plus_errorfor[thisnamek].values.flatten()
            tmpBAclimnon = 10**dfBA_mod_plus_errornon[thisnamek].values.flatten()
            
            tmpBAsst = 10**dfBA_modDT_plus_errorall[thisnamek].values.flatten() # unlog
            tmpBAsstfor = 10**dfBA_modDT_plus_errorfor[thisnamek].values.flatten()
            tmpBAsstnon = 10**dfBA_modDT_plus_errornon[thisnamek].values.flatten()
    
            # calculate the full BA without SST gradient trend influence
            # observed BA - (observed SST in clim-BA model BA - DT SST clim-BA model BA)
            tmpBAnosst = tmpBA - (tmpBAclim - tmpBAsst)
            tmpBAnosstfor = tmpBAfor - (tmpBAclimfor - tmpBAsstfor)
            tmpBAnosstnon = tmpBAnon - (tmpBAclimnon - tmpBAsstnon)
    
            # RULE: BA cannot be negative
            tmpBAnosst[tmpBAnosst<0] = 0
            tmpBAnosstfor[tmpBAnosstfor<0] = 0
            tmpBAnosstnon[tmpBAnosstnon<0] = 0
    
            wumiallDT.append(tmpBAnosst)
            wumiforDT.append(tmpBAnosstfor)
            wuminonDT.append(tmpBAnosstnon)
    
    # transform into pandas dataframe
    wumiallDT = pd.DataFrame(np.transpose(np.stack(wumiallDT)),
                             columns=[nameslist], index=yearxaxis)
    wumiforDT = pd.DataFrame(np.transpose(np.stack(wumiforDT)),
                             columns=[nameslist], index=yearxaxis)
    wuminonDT = pd.DataFrame(np.transpose(np.stack(wuminonDT)),
                             columns=[nameslist], index=yearxaxis)
    
    # --------------

    # SAVE EXPERIMENTS, REPLACE THE CURRENT DICT
    dictBAclimpredall_DT[thisexperiment] = dfsstpredBAall
    dictBAclimpredfor_DT[thisexperiment] = dfsstpredBAfor
    dictBAclimprednon_DT[thisexperiment] = dfsstpredBAnon
    dictBAclimpredall[thisexperiment] = dfclimpredBAall
    dictBAclimpredfor[thisexperiment] = dfclimpredBAfor
    dictBAclimprednon[thisexperiment] = dfclimpredBAnon
    dictwumiallDT[thisexperiment] = wumiallDT
    dictwumiforDT[thisexperiment] = wumiforDT
    dictwuminonDT[thisexperiment] = wuminonDT
    print('for-loop complete for experiment ['+thisexperiment+']')
    
    

for-loop complete for experiment [warmingvarsonly]


for-loop complete for experiment [prioryrwettingonly]


for-loop complete for experiment [y0wettingonly]


In [19]:
# iterate through each experiment to save the data
dfsstpredBAall = [] # reset some variables back to empty
dfsstpredBAfor = []
dfsstpredBAnon = []
dfclimpredBAall = []
dfclimpredBAfor = []
dfclimpredBAnon = []

for j, thisexperiment in enumerate(experimentnames):
    print('output for experiment ['+thisexperiment+']')
    dfsstpredBAall = dictBAclimpredall_DT[thisexperiment]
    dfsstpredBAfor = dictBAclimpredfor_DT[thisexperiment]
    dfsstpredBAnon = dictBAclimprednon_DT[thisexperiment]
    # climate-predicted, from observed climate
    dfclimpredBAall = dictBAclimpredall[thisexperiment]
    dfclimpredBAfor = dictBAclimpredfor[thisexperiment]
    dfclimpredBAnon = dictBAclimprednon[thisexperiment]
    
    dictoutputs = [dfsstpredBAall, dfsstpredBAfor, dfsstpredBAnon,
                   dfclimpredBAall, dfclimpredBAfor, dfclimpredBAnon,
                   wumiallDT, wumiforDT, wuminonDT]
    dictoutputnames = ['sstpred_all', 'sstpred_for', 'sstpred_non',
                       'climpred_all','climpred_for','climpred_non',
                       'wumiDT_all', 'wumiDT_for', 'wumiDT_non']
    # convert to pandas df and export
    for i,thisdict in enumerate(dictoutputs):
        tmpoutput = thisdict # select one of the pd dataframes
        if i<3: # export the SST-predicted
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        if (i>=3) & (i<6): # export the climate-predicted
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        if (i>=6): # export the detrended wumi
            filename = predict_string + 'BurnArea_'+dictoutputnames[i]+'_'+climfilename+'_'+str(firstyear)+'-'+str(finalyear)+'_'+thisexperiment+'.txt'
            tmpoutput.to_csv(filename)
        print('Exported: '+filename)

output for experiment [warmingvarsonly]
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_all_1984-2022_warmingvarsonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_for_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_non_1984-2022_warmingvarsonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_warmingvarsonly.txt
output for experiment [prioryrwettingonly]
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_prioryrwettingonly.txt


...

Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_for_1984-2022_prioryrwettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_non_1984-2022_prioryrwettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_prioryrwettingonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_prioryrwettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_prioryrwettingonly.txt
output for experiment [y0wettingonly]
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_sstpred_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_all_1984-2022_y0wettingonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_for_1984-2022_y0wettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_climpred_non_1984-2022_y0wettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_all_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt


Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_for_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt
Exported: predicted//patch125-155_nino3-34//BurnArea_wumiDT_non_DeTrendClimObs_patch125-155_nino3-34_1984-2022_y0wettingonly.txt


In [20]:
# datetime object containing current date and time
from datetime import datetime
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print("Model last run =", dt_string)

Model last run = 25/06/2026 17:35:16
